In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, mean_squared_error

clf_df = pd.read_csv("classification_escalation.csv")
reg_df = pd.read_csv("resolution_time_regression.csv")

print("Classification shape:", clf_df.shape)
print("Classification columns:", clf_df.columns.tolist())
print("Classification missing values:\n", clf_df.isna().sum(), "\n")

print("Regression shape:", reg_df.shape)
print("Regression columns:", reg_df.columns.tolist())
print("Regression missing values:\n", reg_df.isna().sum())

display(clf_df.head(), reg_df.head())

Classification shape: (900, 7)
Classification columns: ['ticket_id', 'complexity_score', 'team_experience_years', 'priority', 'channel', 'customer_tenure_months', 'escalated']
Classification missing values:
 ticket_id                  0
complexity_score          22
team_experience_years     36
priority                  13
channel                    9
customer_tenure_months    31
escalated                  0
dtype: int64 

Regression shape: (800, 7)
Regression columns: ['ticket_id', 'complexity_score', 'team_experience_years', 'priority', 'channel', 'customer_tenure_months', 'resolution_time_hours']
Regression missing values:
 ticket_id                  0
complexity_score          16
team_experience_years     24
priority                   0
channel                    0
customer_tenure_months    20
resolution_time_hours      0
dtype: int64


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [3]:
# --- EXPERIMENT A: CLASSIFICATION ---

# 1. Separate target (y) and features (X), excluding ticket_id
y_clf = clf_df["escalated"]
X_clf = clf_df.drop(columns=["ticket_id", "escalated"])

# 2. Stratified train/test split (80/20)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=42, stratify=y_clf
)

# Identify numeric and categorical columns automatically
num_cols_c = X_train_c.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_c = X_train_c.select_dtypes(exclude=[np.number]).columns.tolist()

# 3. Leakage-safe preprocessing pipeline
num_pipe_c = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe_c = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_c = ColumnTransformer([
    ("num", num_pipe_c, num_cols_c),
    ("cat", cat_pipe_c, cat_cols_c)
])

# 4. Compare 3 models with identical 5-fold StratifiedKFold CV (F1 metric)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_c = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "k-NN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

cv_results_c = []
for name, clf in models_c.items():
    pipe = Pipeline([
        ("prep", preprocessor_c),
        ("model", clf)
    ])
    scores = cross_validate(pipe, X_train_c, y_train_c, cv=skf, scoring="f1")
    cv_results_c.append({
        "Model": name,
        "CV F1 Mean": round(scores["test_score"].mean(), 4),
        "CV F1 Std": round(scores["test_score"].std(), 4)
    })

cv_df_c = pd.DataFrame(cv_results_c)
print("=== EXPERIMENT A: 5-FOLD STRATIFIED CV RESULTS (F1) ===")
print(cv_df_c.to_string(index=False))

# 5. Tune Logistic Regression using GridSearchCV (you can swap to k-NN/Tree if they win)
tune_pipe_c = Pipeline([
    ("prep", preprocessor_c),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

param_grid_c = {
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__class_weight": [None, "balanced"]
}

grid_c = GridSearchCV(tune_pipe_c, param_grid_c, cv=skf, scoring="f1", n_jobs=-1)
grid_c.fit(X_train_c, y_train_c)

print("\n=== GRIDSEARCHCV BEST MODEL (EXPERIMENT A) ===")
print("Best Params:", grid_c.best_params_)
print("Best CV F1:", round(grid_c.best_score_, 4))

# 6. Final evaluation on the held-out test set ONCE
best_clf = grid_c.best_estimator_
y_pred_c = best_clf.predict(X_test_c)

print("\n=== FINAL TEST SET EVALUATION (EXPERIMENT A) ===")
print("Confusion Matrix (Rows=Actual, Cols=Predicted):")
print(confusion_matrix(y_test_c, y_pred_c))
print("\nClassification Report:")
print(classification_report(y_test_c, y_pred_c))

=== EXPERIMENT A: 5-FOLD STRATIFIED CV RESULTS (F1) ===
              Model  CV F1 Mean  CV F1 Std
Logistic Regression      0.3927     0.0627
               k-NN      0.3608     0.0503
      Decision Tree      0.3307     0.0581

=== GRIDSEARCHCV BEST MODEL (EXPERIMENT A) ===
Best Params: {'model__C': 0.1, 'model__class_weight': 'balanced'}
Best CV F1: 0.52

=== FINAL TEST SET EVALUATION (EXPERIMENT A) ===
Confusion Matrix (Rows=Actual, Cols=Predicted):
[[117  25]
 [ 11  27]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.82      0.87       142
           1       0.52      0.71      0.60        38

    accuracy                           0.80       180
   macro avg       0.72      0.77      0.73       180
weighted avg       0.83      0.80      0.81       180



In [4]:
# --- EXPERIMENT B: REGRESSION ---

# 1. Separate target (y) and features (X), excluding ticket_id
y_reg = reg_df["resolution_time_hours"]
X_reg = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])

# Quick check for outliers in the target
print("=== TARGET (resolution_time_hours) SUMMARY ===")
print(y_reg.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).round(2), "\n")

# 2. Train/test split (80/20)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

# Identify numeric and categorical columns automatically
num_cols_r = X_train_r.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_r = X_train_r.select_dtypes(exclude=[np.number]).columns.tolist()

# 3. Leakage-safe preprocessing pipeline
num_pipe_r = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe_r = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_r = ColumnTransformer([
    ("num", num_pipe_r, num_cols_r),
    ("cat", cat_pipe_r, cat_cols_r)
])

# 4. Compare 3 models using 5-fold CV and MAE
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models_r = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(random_state=42),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42)
}

cv_results_r = []
for name, reg in models_r.items():
    pipe = Pipeline([
        ("prep", preprocessor_r),
        ("model", reg)
    ])
    scores = cross_validate(pipe, X_train_r, y_train_r, cv=kf, scoring="neg_mean_absolute_error")
    mae_scores = -scores["test_score"]
    cv_results_r.append({
        "Model": name,
        "CV MAE Mean": round(mae_scores.mean(), 4),
        "CV MAE Std": round(mae_scores.std(), 4)
    })

cv_df_r = pd.DataFrame(cv_results_r)
print("=== EXPERIMENT B: 5-FOLD CV RESULTS (MAE) ===")
print(cv_df_r.to_string(index=False))

# 5. Tune Ridge alpha using GridSearchCV
tune_pipe_r = Pipeline([
    ("prep", preprocessor_r),
    ("model", Ridge(random_state=42))
])

param_grid_r = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]
}

grid_r = GridSearchCV(tune_pipe_r, param_grid_r, cv=kf, scoring="neg_mean_absolute_error", n_jobs=-1)
grid_r.fit(X_train_r, y_train_r)

best_idx = grid_r.best_index_
best_cv_mae_mean = -grid_r.cv_results_["mean_test_score"][best_idx]
best_cv_mae_std = grid_r.cv_results_["std_test_score"][best_idx]

print("\n=== GRIDSEARCHCV BEST MODEL (EXPERIMENT B) ===")
print("Best Params:", grid_r.best_params_)
print(f"Best CV MAE: {best_cv_mae_mean:.4f} +/- {best_cv_mae_std:.4f} hours")

# 6. Final evaluation on the held-out test set ONCE
best_reg = grid_r.best_estimator_
y_pred_r = best_reg.predict(X_test_r)

test_mae = mean_absolute_error(y_test_r, y_pred_r)
test_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))

print("\n=== FINAL TEST SET EVALUATION (EXPERIMENT B) ===")
print(f"Test MAE:  {test_mae:.4f} hours")
print(f"Test RMSE: {test_rmse:.4f} hours")

=== TARGET (resolution_time_hours) SUMMARY ===
count    800.00
mean       5.87
std        4.14
min        0.50
25%        2.63
50%        5.39
75%        8.35
95%       12.29
99%       18.32
max       27.49
Name: resolution_time_hours, dtype: float64 

=== EXPERIMENT B: 5-FOLD CV RESULTS (MAE) ===
                  Model  CV MAE Mean  CV MAE Std
      Linear Regression       2.0351      0.1717
       Ridge Regression       2.0351      0.1714
Decision Tree Regressor       3.2391      0.0538

=== GRIDSEARCHCV BEST MODEL (EXPERIMENT B) ===
Best Params: {'model__alpha': 0.1}
Best CV MAE: 2.0351 +/- 0.1716 hours

=== FINAL TEST SET EVALUATION (EXPERIMENT B) ===
Test MAE:  2.2301 hours
Test RMSE: 3.4028 hours
